# 03 — Core-Sérsic fit

Fits a Core-Sérsic profile (Graham et al. 2003) — describes massive ellipticals with a depleted central core. We use a log-MSE loss (`log_image_mse`) because the dynamic range spans ~10 decades.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
device, dtype = gl.config.setup(seed=42, device="cpu")


In [ ]:
npix, dx = 128, 0.05
xy = gl.data.coordinate_grid(npix=npix, deltapix=dx)

# Modest CoreSersic params: large central core to keep the simulated
# image away from the (Rb/R)^alpha singularity at R -> 0.
true = dict(Ib=2.0, Re=1.5, Rb=0.4, n=4.0, gamma=0.3, alpha=4.0,
            x0=0.0, y0=0.0, e1=0.10, e2=-0.05)
galaxy = gl.light.CoreSersic(**true)
clean, image = gl.data.simulate_image(
    galaxy, xy,
    psf_fwhm=0.10, deltapix=dx, psf_size=21,
    noise_sigma=0.02, seed=1,
)
ext = (-npix*dx/2, npix*dx/2, -npix*dx/2, npix*dx/2)
gl.viz.side_by_side([clean, image], titles=['clean', 'noisy'],
                    log=True, extent=ext); plt.show()


### Profile cross-section

In [ ]:
r = torch.linspace(0.01, 3.0, 200)
xy_line = torch.stack([r, torch.zeros_like(r)], dim=0)
with torch.no_grad():
    I_core = galaxy(xy_line)
    # comparison: pure Sersic with same Re, n
    pure = gl.light.Sersic(Ie=true['Ib'], Re=true['Re'], n=true['n'],
                           x0=0., y0=0., e1=0., e2=0.)
    I_pure = pure(xy_line)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(r, I_core, 'r-', lw=2, label='core-Sérsic')
ax.plot(r, I_pure, 'k--', lw=1.5, label='Sérsic')
ax.axvline(true['Rb'], color='gray', ls=':', label='$R_b$')
ax.set(xlabel='r [arcsec]', ylabel='I(r)', yscale='log')
ax.legend(); plt.show()


### Fit (log-image MSE)

In [ ]:
class CoreSersicPSF(nn.Module):
    def __init__(self, init):
        super().__init__()
        self.g = gl.light.CoreSersic(**init)
    def forward(self, xy):
        k = gl.light.gaussian_psf_kernel(0.10, dx, size=21)
        return gl.light.convolve_psf(self.g(xy), k)

init = dict(Ib=1.0, Re=1.0, Rb=0.5, n=3.0, gamma=0.5, alpha=3.0,
            x0=0., y0=0., e1=0., e2=0.)
model = CoreSersicPSF(init)

# log_image_mse compresses the 10-decade dynamic range of a galaxy
# surface brightness profile, which makes the gradient finite even
# when the core saturates.
res = gl.inference.fit(
    model, xy, image, gl.inference.log_image_mse,
    lr=0.02, epochs=3000,
    scheduler=gl.inference.optimize.reduce_lr_on_plateau(patience=200, factor=0.7),
    grad_clip=1.0, lbfgs_polish=True,
)
print(f"final loss = {res.best_loss:.4e}")
for k, v in true.items():
    print(f"  {k}: true={v:+.3f}  fit={float(getattr(model.g, k)):+.3f}")


In [ ]:
with torch.no_grad():
    pred = model(xy)
gl.viz.side_by_side([image, pred],
                    titles=['data', 'best fit'], log=True, extent=ext); plt.show()
gl.viz.plot_residuals(image, pred, sigma=0.02); plt.show()
